# Skenario Uji — SIREDO Recommendation Engine

Notebook ini menjalankan **10 skenario uji** terhadap `rekomendasi.engine`
secara terpisah per sel, supaya setiap skenario mudah dijalankan ulang,
dibaca hasilnya, dan didokumentasikan (screenshot/export) satu per satu.

## Struktur notebook
1. **Setup** — import & load data dosen (sekali di awal)
2. **Skenario 1–9** — tiap skenario: 1 sel markdown (tujuan & yang diharapkan)
   + 1 sel kode (jalankan query + cek otomatis + tampilkan tabel top-k)
3. **Skenario 10** — uji performa cache & latency query
4. **Ringkasan akhir** — rekap status semua skenario dalam satu tabel

## Cara pakai
- Jalankan sel **Setup** dulu (sekali saja per sesi).
- Setelah itu, sel skenario manapun bisa dijalankan ulang secara independen
  (asal sel Setup sudah pernah dieksekusi).
- Status **PASS/FAIL** dicek otomatis oleh kode. Status **REVIEW** berarti
  perlu penilaian relevansi secara manual dengan membaca tabel top-k yang
  dicetak.


## 0. Setup

Sesuaikan `PATH_EXCEL_DOSEN` di bawah dengan lokasi file Excel data dosen
Anda. Jika sumber data Anda dari MySQL, ganti isi `muat_data_dosen()`
dengan pemanggilan fungsi loader Anda (lihat komentar OPSI B).


In [ ]:
import json
import time
import statistics
import traceback
from typing import List, Dict, Any, Callable

import pandas as pd

from rekomendasi import engine  # sesuaikan jika nama modul Anda berbeda

# ---------------------------------------------------------
# OPSI B: jika pakai loader MySQL sendiri
# from database.koneksi import get_semua_dosen
# ---------------------------------------------------------

PATH_EXCEL_DOSEN = 'data/dataset_profiles_terintegrasi.xlsx'

def muat_data_dosen() -> List[Dict[str, Any]]:
    print(f"Memuat dataset dosen dari '{PATH_EXCEL_DOSEN}'...")
    df = pd.read_excel(PATH_EXCEL_DOSEN)
    df = df.fillna('-')
    data = df.to_dict('records')
    print(f"   Berhasil memuat {len(data)} baris data dosen dari Excel.")
    return data # type: ignore

data_dosen_asli = muat_data_dosen()


In [ ]:
print("Mempersiapkan cache engine (force_recalculate=False -> pakai cache jika fingerprint cocok)...")
t0 = time.perf_counter()
engine.siapkan_cache(data_dosen_asli, force_recalculate=False)
print(f"Selesai dalam {time.perf_counter() - t0:.2f} detik.")


### Fungsi bantu (jalankan sekali)

Fungsi-fungsi ini dipakai berulang di tiap sel skenario: menjalankan
pipeline, menjalankan cek otomatis, dan mencetak tabel top-k yang rapi.


In [ ]:
import pandas as pd
from IPython.display import display

hasil_semua_skenario = []  # dipakai untuk ringkasan akhir

def cek_skor_wajar(hasil, error=None):
    if error is not None:
        return {"status": "SKIP", "pesan": "Dilewati karena error di tahap eksekusi."}
    rekom = hasil.get("hasil_rekomendasi", [])
    if not rekom:
        return {"status": "REVIEW", "pesan": "Tidak ada hasil rekomendasi sama sekali."}
    for r in rekom:
        for kunci in ("Hybrid Score", "Lexical Score", "Semantic Score"):
            v = r.get(kunci)
            if v is None or v != v:
                return {"status": "FAIL", "pesan": f"{kunci} bernilai None/NaN pada {r.get('NAMA')}"}
            if not (0.0 <= v <= 1.0 + 1e-6):
                return {"status": "FAIL", "pesan": f"{kunci}={v} di luar rentang [0,1] pada {r.get('NAMA')}"}
    return {"status": "PASS", "pesan": "Semua skor dalam rentang [0,1] dan tidak ada NaN."}

def cek_lexical_tidak_selalu_1(hasil, error=None):
    if error is not None:
        return {"status": "SKIP", "pesan": "Dilewati karena error di tahap eksekusi."}
    rekom = hasil.get("hasil_rekomendasi", [])
    if not rekom:
        return {"status": "REVIEW", "pesan": "Tidak ada hasil untuk dicek."}
    skor_lex = [r["Lexical Score"] for r in rekom]
    if all(abs(s - 1.0) < 1e-6 for s in skor_lex):
        return {"status": "FAIL", "pesan": "Semua Lexical Score = 1.0 -> indikasi normalisasi masih over-stretch."}
    return {"status": "PASS", "pesan": f"Distribusi Lexical Score bervariasi: {skor_lex}"}

def cek_tidak_semua_skor_identik(hasil, error=None):
    if error is not None:
        return {"status": "SKIP", "pesan": "Dilewati karena error di tahap eksekusi."}
    rekom = hasil.get("hasil_rekomendasi", [])
    skor_hybrid = [r["Hybrid Score"] for r in rekom]
    if len(set(skor_hybrid)) == 1 and len(skor_hybrid) > 1:
        return {"status": "REVIEW", "pesan": "Semua Hybrid Score identik -> cek apakah ranking jadi arbitrer."}
    return {"status": "PASS", "pesan": f"Sebaran Hybrid Score: {skor_hybrid}"}

def cek_bobot_lex_dilaporkan(hasil, error=None):
    if error is not None:
        return {"status": "SKIP", "pesan": "Dilewati karena error di tahap eksekusi."}
    meta = hasil.get("metadata_mesin", {})
    bl, bs = meta.get("bobot_lex_final"), meta.get("bobot_sem_final")
    if bl is None or bs is None:
        return {"status": "FAIL", "pesan": "bobot_lex_final/bobot_sem_final tidak ada di metadata."}
    return {"status": "REVIEW", "pesan": f"bobot_lex_final={bl}, bobot_sem_final={bs} -> nilai butuh dinilai manual sesuai konteks skenario."}

def cek_tidak_crash_atau_error_jelas(hasil, error=None):
    if error is not None:
        if isinstance(error, (ValueError, RuntimeError)):
            return {"status": "PASS", "pesan": f"Ditolak dengan error jelas: {error}"}
        return {"status": "FAIL", "pesan": f"Crash tak terduga: {type(error).__name__}: {error}"}
    return cek_skor_wajar(hasil)

CEK_MAP = {
    "cek_skor_wajar": cek_skor_wajar,
    "cek_lexical_tidak_selalu_1": cek_lexical_tidak_selalu_1,
    "cek_tidak_semua_skor_identik": cek_tidak_semua_skor_identik,
    "cek_bobot_lex_dilaporkan": cek_bobot_lex_dilaporkan,
    "cek_tidak_crash_atau_error_jelas": cek_tidak_crash_atau_error_jelas,
}

def jalankan_skenario(nama, judul, abstrak, k_rank=5, bobot_lex=-1, cek_list=None):
    """Jalankan satu skenario, cetak tabel top-k, cetak status cek otomatis,
    dan simpan hasilnya ke hasil_semua_skenario untuk ringkasan akhir."""
    cek_list = cek_list or ["cek_skor_wajar"]
    hasil, error = None, None
    t0 = time.perf_counter()
    try:
        hasil = engine.jalankan_pipeline(
            judul_mhs=judul, abstrak_mhs=abstrak, k_rank=k_rank, bobot_lex=bobot_lex,
        )
    except Exception as e:
        error = e
        traceback.print_exc()
    durasi = time.perf_counter() - t0

    print(f"Durasi: {durasi:.3f} detik\n")

    if hasil and hasil.get("hasil_rekomendasi"):
        df = pd.DataFrame(hasil["hasil_rekomendasi"])[
            ["NAMA", "BIDANG_KEAHLIAN", "Hybrid Score", "Lexical Score", "Semantic Score"]
        ]
        display(df)
        meta = hasil.get("metadata_mesin", {})
        print(f"is_adaptif={meta.get('is_adaptif')}  "
              f"bobot_lex_final={meta.get('bobot_lex_final')}  "
              f"bobot_sem_final={meta.get('bobot_sem_final')}")
    else:
        print("(tidak ada hasil_rekomendasi)")

    print()
    status_list = []
    for nama_cek in cek_list:
        r = CEK_MAP[nama_cek](hasil or {}, error)
        status_list.append(r["status"])
        print(f"[{r['status']}] {nama_cek}: {r['pesan']}")

    label = "FAIL" if "FAIL" in status_list else ("REVIEW" if "REVIEW" in status_list else "PASS")
    hasil_semua_skenario.append({"skenario": nama, "status": label, "durasi_detik": round(durasi, 3)})
    print(f"\n>>> STATUS SKENARIO: {label}")
    return hasil


## Skenario 1 — Domain Jelas: Deep Learning Pengenalan Wajah

**Tujuan:** baseline sanity check. **Harapan:** Top-1 harus dosen dengan bidang keahlian AI/ML eksplisit.

In [ ]:
hasil_1_domain_jelas_deep_learning_wajah = jalankan_skenario(
    nama="1_domain_jelas_deep_learning_wajah",
    judul='Penerapan Deep Learning untuk Pengenalan Wajah',
    abstrak='Penelitian ini menggunakan metode Convolutional Neural Network (CNN) untuk mendeteksi wajah secara real-time pada kamera keamanan.',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar', 'cek_lexical_tidak_selalu_1'],
)


## Skenario 2 — Domain Berbeda: Cybersecurity

**Tujuan:** pastikan fix tidak overfit ke satu domain. **Harapan:** Top-1 dosen keamanan siber/jaringan, bukan yang cuma menyebut 'keamanan' sepintas.

In [ ]:
hasil_2_domain_cybersecurity = jalankan_skenario(
    nama="2_domain_cybersecurity",
    judul='Analisis Keamanan Siber pada Aplikasi Web Menggunakan Metode Penetration Testing',
    abstrak='Penelitian ini melakukan pengujian celah keamanan (vulnerability assessment) pada aplikasi web menggunakan metode OWASP dan menganalisis risiko serangan SQL Injection serta Cross-Site Scripting (XSS).',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar', 'cek_lexical_tidak_selalu_1'],
)


## Skenario 3 — Domain Ramai: Sistem Informasi Berbasis Web

**Tujuan:** stress-test saat banyak dosen relevan (topik generik). **Harapan:** ranking antar kandidat mirip tetap masuk akal, bukan random tie.

In [ ]:
hasil_3_domain_ramai_sistem_informasi_web = jalankan_skenario(
    nama="3_domain_ramai_sistem_informasi_web",
    judul='Rancang Bangun Sistem Informasi Manajemen Inventori Berbasis Web',
    abstrak='Sistem ini dibangun menggunakan metode waterfall untuk membantu perusahaan mengelola data inventori barang secara digital berbasis web.',
    k_rank=8,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar', 'cek_tidak_semua_skor_identik'],
)


## Skenario 4 — Domain Niche: Computer Graphics / Animasi 3D

**Tujuan:** pastikan dosen dengan kandidat sedikit (long-tail) tidak dirugikan sistem.

In [ ]:
hasil_4_domain_niche_computer_graphics = jalankan_skenario(
    nama="4_domain_niche_computer_graphics",
    judul='Perancangan Aset 3D dan Animasi Karakter untuk Game Edukasi',
    abstrak='Penelitian ini berfokus pada pembuatan aset grafis 3D low-poly dan rigging animasi karakter menggunakan pipeline computer graphics untuk game edukasi interaktif.',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar'],
)


## 🎯 Skenario 5 — REGRESI: Uji Bug Normalisasi Lama ("kata nyasar")

**Ini uji regresi langsung untuk bug yang sudah diperbaiki.** Proposal ini sengaja meniru judul jurnal seorang dosen non-spesialis kata demi kata. Sebelum fix, kecocokan leksikal literal seperti ini bisa membuat Lexical Score dipaksa ke 1.0 (min-max) dan mendominasi hybrid score meski relevansi semantiknya rendah. **Setelah fix**, top-1 harus tetap dosen AI/ML yang sungguh relevan, bukan dosen yang kebetulan judulnya sama persis tapi keahlian utamanya di bidang lain.

In [ ]:
hasil_5_REGRESI_kata_nyasar = jalankan_skenario(
    nama="5_REGRESI_kata_nyasar",
    judul='Klasifikasi Prompt AI Generative Image Menggunakan Metode Naive Bayes',
    abstrak='Penelitian ini membandingkan performa algoritma Gaussian Naive Bayes dan Decision Tree Classifier dalam klasifikasi prompt AI-generated image.',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar', 'cek_lexical_tidak_selalu_1'],
)


## Skenario 6 — Edge Case: Query Super Pendek (Tanpa Abstrak)

**Harapan:** tidak crash walau abstrak kosong; skor tetap dihasilkan dari judul saja.

In [ ]:
hasil_6_query_super_pendek = jalankan_skenario(
    nama="6_query_super_pendek",
    judul='Sistem Rekomendasi Berbasis Machine Learning',
    abstrak='',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar'],
)


## Skenario 7 — Jargon Teknis Padat

**Harapan:** rasio kata langka tinggi -> `bobot_lex_final` naik signifikan (adaptive weighting), tapi tetap tidak menenggelamkan sinyal semantik yang jelas.

In [ ]:
hasil_7_jargon_padat = jalankan_skenario(
    nama="7_jargon_padat",
    judul='Optimasi Model Vision Transformer untuk Deteksi Objek',
    abstrak='Penelitian menggunakan arsitektur ViT, ResNet, EfficientNet, GAN, dan Autoencoder untuk membandingkan performa deteksi objek dengan pendekatan transfer learning dan fine-tuning hyperparameter.',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar', 'cek_bobot_lex_dilaporkan'],
)


## Skenario 8 — Query Generik Tanpa Jargon

**Harapan (arah sebaliknya dari #7):** nyaris tidak ada jargon -> `bobot_lex_final` turun mendekati batas bawah (0.2), condong ke semantik.

In [ ]:
hasil_8_query_generik_tanpa_jargon = jalankan_skenario(
    nama="8_query_generik_tanpa_jargon",
    judul='Penelitian tentang Peningkatan Layanan di Sebuah Instansi',
    abstrak='Penelitian ini membahas cara meningkatkan kualitas layanan dan kepuasan pengguna di suatu instansi melalui pendekatan yang lebih baik.',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_skor_wajar', 'cek_bobot_lex_dilaporkan'],
)


## Skenario 9 — Edge Case Ekstrem: Query Kosong

**Harapan:** harus di-handle graceful (exception jelas ATAU hasil kosong), bukan crash senyap atau hasil ber-NaN.

In [ ]:
hasil_9_query_kosong = jalankan_skenario(
    nama="9_query_kosong",
    judul='',
    abstrak='   ',
    k_rank=5,
    bobot_lex=-1,
    cek_list=['cek_tidak_crash_atau_error_jelas'],
)


## Skenario 10 — Performa Cache & Latency Query

**Tujuan:** mengukur waktu `siapkan_cache()` cold-start vs cache-hit, dan
konsistensi latency `jalankan_pipeline()` pada beberapa kali query
berturut-turut.

**Harapan:** cache-hit jauh lebih cepat dari cold-start; latency per-query
konsisten (tidak melonjak antar percobaan).


In [ ]:
print("=== COLD START (force_recalculate=True) ===")
t0 = time.perf_counter()
engine.siapkan_cache(data_dosen_asli, force_recalculate=True)
t_cold = time.perf_counter() - t0
print(f"Waktu: {t_cold:.2f} detik")

print("\n=== CACHE-HIT (force_recalculate=False) ===")
t0 = time.perf_counter()
engine.siapkan_cache(data_dosen_asli, force_recalculate=False)
t_warm = time.perf_counter() - t0
print(f"Waktu: {t_warm:.2f} detik")

if t_cold > 0 and t_warm > t_cold * 0.5:
    print("\n⚠ REVIEW: cache-hit tidak jauh lebih cepat dari cold-start. "
          "Cek apakah fingerprint match / tokenisasi BM25 jadi bottleneck.")
else:
    print("\n✅ PASS: cache-hit signifikan lebih cepat dari cold-start.")


In [ ]:
print("=== LATENCY 10x QUERY BERTURUT-TURUT ===")
durasi_query = []
for i in range(10):
    t0 = time.perf_counter()
    engine.jalankan_pipeline(
        judul_mhs="Deep learning untuk klasifikasi citra medis",
        abstrak_mhs="Menggunakan CNN dan transfer learning.",
        k_rank=5,
        bobot_lex=-1,
    )
    durasi_query.append(time.perf_counter() - t0)

print(f"Rata-rata : {statistics.mean(durasi_query):.4f} detik")
print(f"Min       : {min(durasi_query):.4f} detik")
print(f"Max       : {max(durasi_query):.4f} detik")
print(f"Std Dev   : {statistics.pstdev(durasi_query):.4f} detik")

hasil_semua_skenario.append({
    "skenario": "10_performa_cache_dan_latency",
    "status": "REVIEW",
    "durasi_detik": round(statistics.mean(durasi_query), 3),
})


## Ringkasan Akhir

Rekap status semua skenario yang sudah dijalankan pada sesi ini. Jalankan
sel ini paling akhir setelah semua skenario di atas dieksekusi.

**Catatan:** status `PASS` otomatis TIDAK berarti hasil rekomendasi sudah
relevan secara substansi — itu tetap perlu dinilai manusia dengan membaca
tabel top-k di tiap skenario, terutama skenario 1–4 dan skenario regresi #5.


In [ ]:
df_ringkasan = pd.DataFrame(hasil_semua_skenario)
display(df_ringkasan)

total_fail = (df_ringkasan["status"] == "FAIL").sum()
total_review = (df_ringkasan["status"] == "REVIEW").sum()
total_pass = (df_ringkasan["status"] == "PASS").sum()

print(f"\nTotal: {len(df_ringkasan)} skenario | "
      f"{total_fail} FAIL (perlu perbaikan) | "
      f"{total_review} REVIEW (perlu penilaian manual) | "
      f"{total_pass} PASS otomatis penuh")
